# Prompting-strategy effect

Success rate by prompting strategy, per backend.

In [ ]:
from analysis import load_runs

# Pin matplotlib (Agg) + RNG seeds for reproducible, headless figures.
plt = load_runs.pin_style(seed=0)

# Load real runs if any exist, else the deterministic synthetic dataset.
records = load_runs.load_all()
backends = load_runs.backend_names(records)
STRATEGY = "zero_shot"  # the held-fixed strategy for the cross-backend tests
print(f"{len(records)} runs, backends={backends}")

In [ ]:
from collections import defaultdict

import numpy as np

from analysis import stats as st
from analysis.aggregate import aggregate_runs, build_outcome_matrix

summaries = aggregate_runs(records)
strategies = load_runs.strategies(records)
rate = defaultdict(dict)  # backend -> strategy -> success rate
for strat in strategies:
    ids, matrix = build_outcome_matrix(summaries, backends=backends, strategy=strat)
    n = len(ids)
    for j, b in enumerate(backends):
        rate[b][strat] = (sum(row[j] for row in matrix) / n) if n else 0.0
for b in backends:
    print(b, {s: round(rate[b][s], 2) for s in strategies})

In [ ]:
# Cochran's Q with prompting strategy as the within-subject factor, for one
# backend (binaries are the subjects, strategies the conditions).
focus = backends[0]
by_binary = defaultdict(dict)
for s in summaries:
    if s.backend_name == focus:
        by_binary[s.binary_id][s.strategy] = s.success
ordered = sorted(by_binary)
strat_matrix = [[by_binary[b].get(strat, 0) for strat in strategies] for b in ordered]
q_strategy = st.cochrans_q(strat_matrix)
print(f"Cochran's Q over strategies for {focus}: {q_strategy}")

In [ ]:
x = np.arange(len(strategies))
width = 0.8 / max(1, len(backends))
fig, ax = plt.subplots()
for i, b in enumerate(backends):
    ax.bar(x + i * width, [rate[b][s] for s in strategies], width, label=b)
ax.set_xticks(x + width * (len(backends) - 1) / 2)
ax.set_xticklabels(strategies, rotation=15, ha="right")
ax.set_ylabel("success rate")
ax.set_ylim(0, 1)
ax.set_title("Prompting strategy effect by backend")
ax.legend(fontsize=8)
load_runs.save_figure(fig, "07_strategy_effect", "strategy_effect")
fig

In [ ]:
from IPython.display import Markdown

best = {b: max(strategies, key=lambda s: rate[b][s]) for b in backends}
lines = [
    f"- **{b}**: best strategy **{best[b]}** "
    f"({rate[b][best[b]]:.2f}); " + ", ".join(f"{s}={rate[b][s]:.2f}" for s in strategies)
    for b in backends
]
verdict = "reject H0" if q_strategy.p_value < 0.05 else "fail to reject H0"
Markdown(
    "### Summary — prompting-strategy effect\n\n"
    + "\n".join(lines)
    + f"\n\nCochran's Q over strategies for **{focus}**: "
    f"Q = {q_strategy.statistic:.3f}, p = {q_strategy.p_value:.4g} ({verdict})."
)